# Task 2 — Classificatori manuali su `manuale.csv`

**Corso:** Fondamenti e Applicazioni del Machine Learning (FML 2026)

La traccia chiede di **definire manualmente** uno o due classificatori (due, perché
il gruppo è di due persone), illustrare i passi per **adattarli ai dati**,
**implementarli in Python** e **valutarne le prestazioni** sullo stesso `manuale.csv`.

I due modelli scelti, entrambi dalla **Lezione 5** del corso:

| Modello | Idea | Sfrutta |
|---------|------|---------|
| **1R (1-Rule)** | una regola basata su **un solo** attributo | l'attributo più discriminante |
| **Naïve Bayes** | combina **tutti** gli attributi con le probabilità | nominali (frequenze) + numerici (gaussiana) |

Insieme raccontano una **progressione logica**: 1R è il baseline minimo (un solo
attributo), Naïve Bayes il passo successivo (tutti gli attributi, in chiave
probabilistica).

**Riferimenti:** Witten et al., *Data Mining* (4ª ed.), cap. 4 — Lezione 5.


## 0. Caricamento dei dati

In [1]:
import pandas as pd
import numpy as np
from collections import Counter

In [2]:
m = pd.read_csv("../data/processed/manuale.csv")
m

,age,campaign,job,marital,education,housing,loan,contact,poutcome,y
0,35,3,admin.,single,professional.course,yes,no,cellular,nonexistent,1
1,53,1,blue-collar,married,unknown,no,no,cellular,nonexistent,0
2,36,3,blue-collar,married,basic.9y,yes,no,cellular,failure,0
3,61,1,retired,married,basic.4y,no,no,telephone,nonexistent,1
4,47,3,admin.,married,university.degree,no,no,telephone,nonexistent,0
5,36,4,blue-collar,married,unknown,yes,no,cellular,nonexistent,0
6,30,4,blue-collar,married,basic.6y,yes,no,cellular,nonexistent,0
7,31,1,admin.,single,high.school,yes,no,cellular,nonexistent,1
8,51,5,technician,married,university.degree,yes,no,cellular,nonexistent,1
9,35,1,blue-collar,divorced,basic.9y,no,no,cellular,failure,1


In [3]:
nominali = ["job", "marital", "education", "housing", "loan", "contact", "poutcome"]
numerici = ["age", "campaign"]
print("Nominali:", nominali)
print("Numerici:", numerici)
print("\nDistribuzione classe:")
print(m["y"].value_counts())

Nominali: ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'poutcome']
Numerici: ['age', 'campaign']

Distribuzione classe:
y
1    6
0    6
Name: count, dtype: int64


---
# Parte A — 1R (1-Rule)

## A.1 Come funziona (teoria, Lezione 5)

1R costruisce un **albero decisionale a un solo livello**: usa un unico attributo
per classificare. La procedura:

1. **per ogni attributo**, si genera una regola: per ciascun valore dell'attributo,
   si guarda quale **classe è più frequente** tra le istanze con quel valore, e si
   assegna quella classe;
2. si **contano gli errori** che la regola commette sul training set (le istanze la
   cui classe reale è diversa da quella assegnata dalla regola);
3. si **sceglie l'attributo** la cui regola produce il **minor numero di errori**.

Per gli **attributi numerici** (`age`, `campaign`) 1R richiede una
**discretizzazione**: i valori continui vengono divisi in intervalli (*bin*).
Qui usiamo un binning semplice sulla **mediana** (due intervalli).

> **Punto critico (dalle slide):** 1R tende all'**overfitting** quando un attributo
> ha **molti valori distinti**. Al limite, un attributo con un valore diverso per
> ogni istanza avrebbe 0 errori ma sarebbe inutile. Lo vedremo concretamente.


## A.2 Adattamento ai dati — calcolo degli errori per ogni attributo

Definiamo la funzione che, dato un attributo (già nominale o discretizzato),
costruisce la regola 1R e conta gli errori.


In [4]:
def regola_1R(serie_attr, y):
    """Costruisce la regola 1R per un attributo e restituisce (regole, errori_totali).
    regole: dizionario {valore: classe_assegnata}."""
    regole = {}
    errori = 0
    for val in serie_attr.unique():
        mask = (serie_attr == val)
        classi = y[mask]
        classe_magg = classi.mode()[0]          # classe a maggioranza per quel valore
        regole[val] = classe_magg
        errori += (classi != classe_magg).sum()  # errori = istanze non a maggioranza
    return regole, int(errori)

### A.2.1 Errori sugli attributi nominali

In [5]:
errori_attr = {}
for attr in nominali:
    regole, err = regola_1R(m[attr], m["y"])
    errori_attr[attr] = err
    print(f"{attr:12s}: {err}/12 errori   regole={regole}")

job         : 2/12 errori   regole={'admin.': np.int64(1), 'blue-collar': np.int64(0), 'retired': np.int64(1), 'technician': np.int64(1), 'student': np.int64(1), 'unknown': np.int64(0)}
marital     : 2/12 errori   regole={'single': np.int64(1), 'married': np.int64(0), 'divorced': np.int64(1)}
education   : 3/12 errori   regole={'professional.course': np.int64(1), 'unknown': np.int64(0), 'basic.9y': np.int64(1), 'basic.4y': np.int64(1), 'university.degree': np.int64(0), 'basic.6y': np.int64(0), 'high.school': np.int64(0)}
housing     : 5/12 errori   regole={'yes': np.int64(0), 'no': np.int64(0), 'unknown': np.int64(1)}
loan        : 5/12 errori   regole={'no': np.int64(0), 'unknown': np.int64(1), 'yes': np.int64(0)}
contact     : 5/12 errori   regole={'cellular': np.int64(1), 'telephone': np.int64(0)}
poutcome    : 6/12 errori   regole={'nonexistent': np.int64(0), 'failure': np.int64(0)}


### A.2.2 Errori sugli attributi numerici (discretizzati per mediana)

In [6]:
for attr in numerici:
    med = m[attr].median()
    binned = (m[attr] > med).map({False: f"<= {med}", True: f"> {med}"})
    regole, err = regola_1R(binned, m["y"])
    errori_attr[attr] = err
    print(f"{attr:12s} (mediana={med}): {err}/12 errori   regole={regole}")

age          (mediana=36.0): 5/12 errori   regole={'<= 36.0': np.int64(1), '> 36.0': np.int64(0)}
campaign     (mediana=2.5): 4/12 errori   regole={'> 2.5': np.int64(0), '<= 2.5': np.int64(1)}


### A.2.3 Scelta dell'attributo migliore

In [7]:
print("Errori per attributo:")
for a, e in sorted(errori_attr.items(), key=lambda x: x[1]):
    print(f"  {a:12s}: {e}/12")

best = min(errori_attr, key=errori_attr.get)
print(f"\n=> 1R sceglie '{best}' con {errori_attr[best]}/12 errori")
print(f"   Accuratezza sul training: {(12-errori_attr[best])/12:.2%}")

Errori per attributo:
  job         : 2/12
  marital     : 2/12
  education   : 3/12
  campaign    : 4/12
  housing     : 5/12
  loan        : 5/12
  contact     : 5/12
  age         : 5/12
  poutcome    : 6/12

=> 1R sceglie 'job' con 2/12 errori
   Accuratezza sul training: 83.33%


> **Osservazioni critiche (da spiegare all'orale):**
>
> - C'è un **pareggio** tra `job` e `marital` (entrambi 2 errori). 1R sceglie il
>   primo in ordine, ma è un caso interessante di *tie-breaking*.
> - `job` ha però **4 valori con una sola istanza** (retired, technician, student,
>   unknown): ognuno ottiene 0 errori "gratis". È proprio l'**overfitting** che le
>   slide segnalano per gli attributi a molti valori.
> - `marital` ha solo **3 valori** ed è più robusto: la regola "single→1, married→0,
>   divorced→1" si generalizza meglio. In un contesto reale sarebbe preferibile a
>   `job`, pur avendo lo stesso numero di errori sul training.


## A.3 Implementazione: il classificatore 1R completo

Mettiamo insieme i pezzi in una funzione che addestra 1R (sceglie l'attributo e la
regola) e una che predice.


In [8]:
def addestra_1R(df, attributi_nominali, attributi_numerici, y_col="y"):
    errori_attr, regole_attr, soglie = {}, {}, {}
    # nominali
    for attr in attributi_nominali:
        reg, err = regola_1R(df[attr], df[y_col])
        errori_attr[attr] = err; regole_attr[attr] = reg
    # numerici (binning su mediana)
    for attr in attributi_numerici:
        med = df[attr].median(); soglie[attr] = med
        binned = (df[attr] > med).map({False: "low", True: "high"})
        reg, err = regola_1R(binned, df[y_col])
        errori_attr[attr] = err; regole_attr[attr] = reg
    best = min(errori_attr, key=errori_attr.get)
    return {"attributo": best, "regole": regole_attr[best],
            "soglia": soglie.get(best), "errori": errori_attr[best]}

In [9]:
def predici_1R(modello, istanza):
    attr = modello["attributo"]
    if modello["soglia"] is not None:          # attributo numerico
        val = "high" if istanza[attr] > modello["soglia"] else "low"
    else:
        val = istanza[attr]
    return modello["regole"].get(val, 0)        # default 0 se valore mai visto

### A.3.1 Valutazione in leave-one-out

Con sole 12 istanze usiamo il **leave-one-out** (addestrare su 11, testare su 1):
è il modo corretto di stimare le prestazioni su pochi dati ed è il caso estremo
della cross-validation (Lezione 8).


In [10]:
pred_1R = []
for i in range(len(m)):
    train_i = m.drop(i).reset_index(drop=True)
    modello = addestra_1R(train_i, nominali, numerici)
    pred_1R.append(predici_1R(modello, m.iloc[i]))

acc_1R = (np.array(pred_1R) == m["y"].values).mean()
print(f"Accuratezza 1R (leave-one-out): {acc_1R:.2%}")

Accuratezza 1R (leave-one-out): 41.67%


---
# Parte B — Naïve Bayes

## B.1 Come funziona (teoria, Lezione 5)

Naïve Bayes si basa sulla **regola di Bayes**. Per un'istanza con attributi
$x_1, \dots, x_n$:

$$ P(c \mid x_1,\dots,x_n) \propto P(c)\cdot \prod_{i=1}^{n} P(x_i \mid c) $$

- $P(c)$ = probabilità **a priori** della classe;
- $P(x_i \mid c)$ = **verosimiglianza** dell'attributo data la classe;
- il prodotto assume l'**indipendenza** degli attributi (da qui "naïve").

Si sceglie la classe col prodotto più alto (regola **MAP**).

**Due tipi di attributo:**
- **nominali** → frequenze, con **stimatore di Laplace** (conteggi inizializzati a 1)
  per evitare probabilità nulle:
  $$ P(x_i \mid c) = \frac{(\text{conteggio} ) + 1}{N_c + v_i} $$
- **numerici** → **distribuzione gaussiana** con media e deviazione standard stimate:
  $$ P(x_i \mid c) = \frac{1}{\sqrt{2\pi}\,\sigma}\, e^{-\frac{(x_i-\mu)^2}{2\sigma^2}} $$


## B.2 Adattamento ai dati — calcolo **a mano** su un'istanza

Prendiamo come test la **riga 0**, addestrando sulle altre 11.


In [11]:
test = m.iloc[0]
train = m.drop(0).reset_index(drop=True)
print("Istanza di test (riga 0):")
print(test)
print("\nClasse reale:", int(test["y"]))

Istanza di test (riga 0):
age                           35
campaign                       3
job                       admin.
marital                   single
education    professional.course
housing                      yes
loan                          no
contact                 cellular
poutcome             nonexistent
y                              1
Name: 0, dtype: object

Classe reale: 1


### B.2.1 Probabilità a priori

In [12]:
n = len(train)
n1 = int((train["y"] == 1).sum()); n0 = int((train["y"] == 0).sum())
print(f"P(y=1) = {n1}/{n} = {n1/n:.4f}")
print(f"P(y=0) = {n0}/{n} = {n0/n:.4f}")

P(y=1) = 5/11 = 0.4545
P(y=0) = 6/11 = 0.5455


### B.2.2 Verosimiglianze nominali (stimatore di Laplace)

In [13]:
def likelihood_nom(attr, val, cls):
    sub = train[train["y"] == cls]
    v = train[attr].nunique()
    count = int((sub[attr] == val).sum())
    return (count + 1) / (len(sub) + v), count, len(sub), v

In [14]:
for cls in [1, 0]:
    print(f"--- Classe y={cls} ---")
    for attr in nominali:
        p, c, Nc, v = likelihood_nom(attr, test[attr], cls)
        print(f"  P({attr}={test[attr]}|y={cls}) = ({c}+1)/({Nc}+{v}) = {p:.4f}")
    print()

--- Classe y=1 ---
  P(job=admin.|y=1) = (1+1)/(5+6) = 0.1818
  P(marital=single|y=1) = (2+1)/(5+3) = 0.3750
  P(education=professional.course|y=1) = (0+1)/(5+6) = 0.0909
  P(housing=yes|y=1) = (2+1)/(5+3) = 0.3750
  P(loan=no|y=1) = (4+1)/(5+3) = 0.6250
  P(contact=cellular|y=1) = (4+1)/(5+2) = 0.7143
  P(poutcome=nonexistent|y=1) = (4+1)/(5+2) = 0.7143

--- Classe y=0 ---
  P(job=admin.|y=0) = (1+1)/(6+6) = 0.1667
  P(marital=single|y=0) = (0+1)/(6+3) = 0.1111
  P(education=professional.course|y=0) = (0+1)/(6+6) = 0.0833
  P(housing=yes|y=0) = (4+1)/(6+3) = 0.5556
  P(loan=no|y=0) = (5+1)/(6+3) = 0.6667
  P(contact=cellular|y=0) = (4+1)/(6+2) = 0.6250
  P(poutcome=nonexistent|y=0) = (5+1)/(6+2) = 0.7500



### B.2.3 Verosimiglianze numeriche (gaussiana)

In [15]:
def likelihood_num(attr, val, cls):
    sub = train[train["y"] == cls][attr]
    mu, sigma = sub.mean(), sub.std(ddof=1)
    p = (1 / (np.sqrt(2*np.pi) * sigma)) * np.exp(-((val - mu)**2) / (2 * sigma**2))
    return p, mu, sigma

In [16]:
for cls in [1, 0]:
    print(f"--- Classe y={cls} ---")
    for attr in numerici:
        p, mu, sig = likelihood_num(attr, test[attr], cls)
        print(f"  P({attr}={test[attr]}|y={cls}): mu={mu:.2f}, sigma={sig:.2f} -> densita={p:.5f}")
    print()

--- Classe y=1 ---
  P(age=35|y=1): mu=39.80, sigma=16.04 -> densita=0.02379
  P(campaign=3|y=1): mu=1.80, sigma=1.79 -> densita=0.17808

--- Classe y=0 ---
  P(age=35|y=0): mu=40.83, sigma=8.42 -> densita=0.03726
  P(campaign=3|y=0): mu=2.83, sigma=1.17 -> densita=0.33780



### B.2.4 Combinazione: posterior proporzionale

In [17]:
def naive_bayes_score(istanza, train_df):
    n = len(train_df); scores = {}
    for cls in [0, 1]:
        sub = train_df[train_df["y"] == cls]
        score = len(sub) / n
        for attr in nominali:
            v = train_df[attr].nunique()
            count = int((sub[attr] == istanza[attr]).sum())
            score *= (count + 1) / (len(sub) + v)
        for attr in numerici:
            mu, sigma = sub[attr].mean(), sub[attr].std(ddof=1)
            score *= (1/(np.sqrt(2*np.pi)*sigma)) * np.exp(-((istanza[attr]-mu)**2)/(2*sigma**2))
        scores[cls] = score
    return scores

In [18]:
scores = naive_bayes_score(test, train)
print(f"score(y=0) = {scores[0]:.3e}")
print(f"score(y=1) = {scores[1]:.3e}")
pred = max(scores, key=scores.get)
print(f"\nPredizione: y={pred}  |  Classe reale: y={int(test['y'])}")
print("ESITO:", "corretto" if pred == test["y"] else "ERRATO")

score(y=0) = 1.839e-06
score(y=1) = 1.427e-06

Predizione: y=0  |  Classe reale: y=1
ESITO: ERRATO


> **Osservazione.** Su questa istanza Naïve Bayes predice `y=0`, mentre la classe
> reale è `y=1`: un errore. Con così pochi dati le stime di probabilità sono
> instabili e alcuni attributi nominali "tirano" verso la classe 0.


## B.3 Implementazione completa — leave-one-out

In [19]:
pred_nb = []
for i in range(len(m)):
    s = naive_bayes_score(m.iloc[i], m.drop(i).reset_index(drop=True))
    pred_nb.append(max(s, key=s.get))

acc_nb = (np.array(pred_nb) == m["y"].values).mean()
print(f"Accuratezza Naive Bayes (leave-one-out): {acc_nb:.2%}")

Accuratezza Naive Bayes (leave-one-out): 50.00%


## B.4 Confronto con l'API di Scikit-Learn

La traccia consente l'uso di API. Usiamo `GaussianNB` come controprova (sklearn
tratta i nominali come numeri, quindi prima li codifichiamo: il risultato non sarà
identico al nostro, che distingue nominali e numerici).


In [20]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import LeaveOneOut

X = m[numerici + nominali].copy()
X[nominali] = OrdinalEncoder().fit_transform(X[nominali])
y = m["y"].values

preds = []
for tr, te in LeaveOneOut().split(X):
    clf = GaussianNB().fit(X.iloc[tr], y[tr])
    preds.append(clf.predict(X.iloc[te])[0])
print(f"Accuratezza GaussianNB (sklearn, LOO): {(np.array(preds)==y).mean():.2%}")

Accuratezza GaussianNB (sklearn, LOO): 58.33%


---
# Riepilogo e analisi critica

Abbiamo definito **a mano** e implementato in Python due classificatori della
Lezione 5:

| Modello | Accuratezza (LOO) | Note |
|---------|-------------------|------|
| **1R** | vedi output A.3.1 | usa il solo attributo `job` (pareggio con `marital`) |
| **Naïve Bayes** | vedi output B.3 | combina tutti gli attributi |

**Punti di discussione per l'orale:**
- Le metriche su **12 istanze** sono poco affidabili: servono a illustrare il
  funzionamento, non a giudicare i modelli (la valutazione seria è nei Task 4–5).
- **1R** è semplicissimo e interpretabile, ma soffre di **overfitting** con attributi
  a molti valori (caso `job`): un attributo più "povero" come `marital` sarebbe più
  robusto a parità di errori.
- **Naïve Bayes** sfrutta più informazione (tutti gli attributi) ma assume
  **indipendenza** e **normalità** dei numerici: assunzioni forti su pochi dati.
- Confrontare 1R e Naïve Bayes mostra il **trade-off semplicità/ricchezza**: un solo
  attributo contro tutti.

**Prossimo passo (Task 3):** analisi esplorativa di `training.csv` (boxplot,
pairplot, matrice di correlazione).
